# 02 — Depth is a representation hierarchy

In notebook 1 you built a complete network by hand: one hidden layer, a softmax head, a training
loop. It already bends decision boundaries. This notebook asks the question the whole chapter is
built around: what does **stacking many layers** buy you?

We will *see* the answer first — watch a deep network remap a tangled 2-D problem until a straight
line can split it — and then *measure* it honestly on real data, where the story is more nuanced
than the hype suggests.

**Prerequisites**
- Notebook 1 — the network we built by hand (forward pass, backward pass, the training loop).
- Chapter 00 — the train/test split, accuracy, feature scaling.
- Chapter 03 — linear separability: what a straight line can and cannot split.

**What you'll be able to do**
- Generalize the notebook-1 network to **any number of layers**.
- *See* successive layers remap a problem into a linearly separable one.
- *Measure* depth's honest gain — and recognize when width, not depth, is doing the work.
- Name *why* the vivid feature hierarchy needs more than a dense network.

## One layer was good — why go deeper?

Notebook 1's network had a single hidden layer: `2 → 16 → 3`. It learned curved boundaries and
generalized well. So why does the field talk endlessly about **deep** networks?

There are two ways to grow a network:

- **width** — more units in a layer (16 → 256);
- **depth** — more layers stacked one after another (one hidden layer → five).

This notebook is about the second. The bet of deep learning is that **depth** — composing many
transformations — is an especially powerful way to spend a network's capacity. Let's find out what
that bet actually buys, and be honest about when it pays.

## A layer remaps; depth composes remaps

Here is the key picture. A hidden layer takes the input and **transforms it into a new
representation** — it stretches, folds, and bends the space (that is what the weights plus a
non-linearity do, chapter 11). The next layer does not see the original input at all; it sees that
*new* representation, and remaps it again.

So a deep network is a **composition** of remaps:

`input → representation 1 → representation 2 → ... → a decision`

The bet is that each remap makes the problem a little easier, until the final layer faces something
so plain — ideally **linearly separable** — that a single straight cut solves it. "Deep" means many
layers in that chain.

Let's watch it happen.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import fetch_openml, make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from ml_course import colors, viz

viz.use_course_style()

SEED = 0


# An L-layer network: notebook 1's forward/backward, now a loop over a list of layers.
# Each layer is a weight matrix W and a bias b. The hidden activation is a CHOICE (ReLU or
# tanh); the output is always softmax + cross-entropy, exactly as in notebook 1.
def init_params(sizes, scale=0.1, seed=SEED):
    # sizes = [n_in, h1, h2, ..., n_out]. Small random init breaks symmetry (chapter 11);
    # the principled way to choose the scale is notebook 4.
    rng = np.random.default_rng(seed)
    params = []
    for n_in, n_out in zip(sizes[:-1], sizes[1:], strict=True):
        params.append({"W": rng.standard_normal((n_in, n_out)) * scale, "b": np.zeros(n_out)})
    return params


def softmax(z):
    z = z - z.max(axis=1, keepdims=True)  # subtract the row max for numerical stability
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def forward(params, X, hidden="relu"):
    # Return ALL activations [X, a1, a2, ..., probs] so we can both train and look at
    # what each layer represents.
    acts = [X]
    a = X
    last = len(params) - 1
    for i, layer in enumerate(params):
        z = a @ layer["W"] + layer["b"]
        a = softmax(z) if i == last else (np.maximum(0.0, z) if hidden == "relu" else np.tanh(z))
        acts.append(a)
    return acts


def cross_entropy(probs, y):
    n = len(y)
    return float(-np.mean(np.log(probs[np.arange(n), y] + 1e-12)))


def backward(params, acts, y, hidden="relu"):
    # The chain rule of chapter 11 / notebook 1, iterated from the last layer to the first.
    n = len(y)
    probs = acts[-1]
    y_onehot = np.zeros_like(probs)
    y_onehot[np.arange(n), y] = 1.0
    d = (probs - y_onehot) / n  # softmax + cross-entropy gradient (the (p - y)/n of notebook 1)
    grads = [None] * len(params)
    for i in reversed(range(len(params))):
        grads[i] = {"W": acts[i].T @ d, "b": d.sum(axis=0)}
        if i > 0:
            da = d @ params[i]["W"].T
            d = da * (acts[i] > 0) if hidden == "relu" else da * (1.0 - acts[i] ** 2)
    return grads


def train(sizes, X, y, *, hidden="relu", lr, epochs, batch_size=None, seed=SEED, track_loss=False):
    # Gradient descent (chapter 03 / notebook 1). batch_size=None means full-batch.
    params = init_params(sizes, seed=seed)
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    bs = n if batch_size is None else batch_size
    losses = []
    for _ in range(epochs):
        if track_loss:
            losses.append(cross_entropy(forward(params, X, hidden)[-1], y))
        order = rng.permutation(n)  # shuffle once per epoch, then iterate contiguous batches
        for start in range(0, n, bs):
            batch = order[start:start + bs]
            acts = forward(params, X[batch], hidden)
            grads = backward(params, acts, y[batch], hidden)
            for i in range(len(params)):
                params[i]["W"] -= lr * grads[i]["W"]
                params[i]["b"] -= lr * grads[i]["b"]
    return params, losses


print("L-layer net ready: init_params / forward / cross_entropy / backward / train")
print("hidden activation is a choice: 'relu' or 'tanh'")

## It is notebook 1's machinery, in a loop

Read the code above: there is nothing new in it. The forward pass is the same matrix-multiply plus
activation as notebook 1, now repeated for each layer in a list. The backward pass is the same chain
rule, now walked from the last layer back to the first. The gradient at the output is the very same
softmax + cross-entropy gradient `(p − y) / n` you derived in notebook 1. We **package** it so it
works for any depth — we do not re-derive it.

Two small generalizations: the hidden activation is now a *choice* (`relu` or `tanh`), because we
will want both — `tanh` for a tiny problem we can visualize, `relu` for the larger one — and `train`
takes a `batch_size`. With `batch_size=None` it is the full-batch descent of notebook 1; with a size
set, it does **mini-batch** gradient descent — shuffle the data each epoch, then take one step per
small slice (chapter 11's vocabulary; notebook 8 returns to it).

As always in this course, we trust a by-hand gradient only after **checking** it against finite
differences.

In [ ]:
# Gradient-check the L-layer net (the chapter-11 habit): nudge each weight by a tiny eps,
# see how the loss moves, and compare that to our analytic gradient.
def numerical_grad(params, X, y, li, idx, hidden, eps=1e-6):
    flat = params[li]["W"].ravel()
    original = flat[idx]
    flat[idx] = original + eps
    loss_plus = cross_entropy(forward(params, X, hidden)[-1], y)
    flat[idx] = original - eps
    loss_minus = cross_entropy(forward(params, X, hidden)[-1], y)
    flat[idx] = original
    return (loss_plus - loss_minus) / (2 * eps)


X_chk, y_chk = make_moons(n_samples=50, noise=0.2, random_state=SEED)
chk = init_params([2, 4, 4, 2], seed=3)  # a 4-layer net to exercise the whole chain
grads_chk = backward(chk, forward(chk, X_chk, "tanh"), y_chk, "tanh")

gen = np.random.default_rng(0)
max_rel_error = 0.0
for li in range(len(chk)):
    g = grads_chk[li]["W"].ravel()
    for idx in gen.choice(g.size, size=min(8, g.size), replace=False):
        numeric = numerical_grad(chk, X_chk, y_chk, li, idx, "tanh")
        denom = max(1e-12, abs(g[idx]) + abs(numeric))
        max_rel_error = max(max_rel_error, abs(g[idx] - numeric) / denom)
print(f"largest relative gradient error (4-layer net) = {max_rel_error:.2e}")
print("on the order of 1e-7 (finite-difference precision at this step): correct at any depth.")

## Seeing a remap: untangling two moons

To *watch* depth work, we need a problem we can plot at every layer. We use **two moons** — two
interleaving crescents. A straight line cannot separate them (we will measure exactly how badly), so
the network has to remap them into something a line *can* split.

The trick that lets us see it: build an **all-width-2** network, `2 → 2 → 2 → 2`. Every layer has two
units, so every layer's activation is a point in the plane and we can scatter it — two hidden `tanh`
layers, then a 2-class softmax head. (For two classes, a two-output softmax is the same model as a
single sigmoid neuron — chapter 11 — written as softmax so the machinery stays identical to
notebook 1's.)

In [ ]:
# Two moons: interleaving crescents. Standardize (chapter 00).
X_moons, y_moons = make_moons(n_samples=400, noise=0.20, random_state=SEED)
X_moons = StandardScaler().fit_transform(X_moons)


def linear_separability(representation, y):
    # How well can the BEST straight line split this representation? Fit a linear classifier with
    # regularization turned off (large C), so we measure the best a line can do, then read its
    # accuracy. A probe ON the representation, not the model we are studying.
    probe = LogisticRegression(C=1e6, max_iter=10000)
    return probe.fit(representation, y).score(representation, y)


# All-width-2 net: 2 -> 2 -> 2 -> 2 (two hidden tanh layers + a 2-class softmax head).
deep_narrow, losses = train(
    [2, 2, 2, 2], X_moons, y_moons, hidden="tanh", lr=0.5, epochs=8000, track_loss=True
)
acts_moons = forward(deep_narrow, X_moons, hidden="tanh")
acc_moons = (acts_moons[-1].argmax(1) == y_moons).mean()

print(f"loss: {losses[0]:.3f} -> {losses[-1]:.3f}   (ln 2 = {np.log(2):.3f})")
print(f"net accuracy = {acc_moons:.3f}")
print("linear separability, layer by layer:")
print(f"  input    : {linear_separability(X_moons, y_moons):.3f}")
print(f"  hidden 1 : {linear_separability(acts_moons[1], y_moons):.3f}")
print(f"  hidden 2 : {linear_separability(acts_moons[2], y_moons):.3f}")

# For contrast: a single WIDE layer (2 -> 8 -> 2) also untangles the moons.
wide_one, _ = train([2, 8, 2], X_moons, y_moons, hidden="tanh", lr=0.5, epochs=8000)
acc_wide = (forward(wide_one, X_moons, "tanh")[-1].argmax(1) == y_moons).mean()
print(f"\none wide layer (2 -> 8 -> 2): net accuracy = {acc_wide:.3f}")

In [ ]:
stages = [
    ("Input (the two features)", X_moons),
    ("After hidden layer 1", acts_moons[1]),
    ("After hidden layer 2", acts_moons[2]),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
for ax, (title, rep) in zip(axes, stages, strict=True):
    for k in (0, 1):
        mask = y_moons == k
        ax.scatter(
            rep[mask, 0], rep[mask, 1],
            color=colors.CLASS_CYCLE[k], edgecolor=colors.COLORS["text"],
            linewidth=0.4, s=25, label=f"class {k}",
        )
    ax.set_title(title)
    ax.set_xlabel("coordinate 1")
    ax.set_ylabel("coordinate 2")
axes[0].legend()
fig.suptitle("Each layer remaps the data — the moons untangle", y=1.01)
fig.tight_layout()
plt.show()

**Read the figure.** Left: the input — two crescents that interlock, so no straight line separates
the colours. Middle: after the **first** hidden layer the space has been bent enough that the two
classes already fall into groups a straight line can split. Right: the second hidden layer
consolidates them a little further. The network did not memorize a boundary; it **remapped the data**
until a straight cut was enough — which is exactly what the softmax output layer then makes.

In [ ]:
layer_names = ["input", "hidden 1", "hidden 2"]
sep_values = [
    linear_separability(X_moons, y_moons),
    linear_separability(acts_moons[1], y_moons),
    linear_separability(acts_moons[2], y_moons),
]

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(
    layer_names, sep_values, color=colors.CLASS_CYCLE[3],
    edgecolor=colors.COLORS["text"], linewidth=0.6,
)
for bar, value in zip(bars, sep_values, strict=True):
    ax.text(
        bar.get_x() + bar.get_width() / 2, value, f"{value:.3f}",
        ha="center", va="bottom", color=colors.COLORS["text"],
    )
ax.axhline(0.5, color=colors.COLORS["muted"], ls="--", lw=1, label="chance (0.5)")
ax.set_ylim(0, 1.08)
ax.set_ylabel("linear-separability accuracy")
ax.set_title("How linearly separable is each layer's representation?")
ax.legend()
plt.show()

**Read the figure.** We put a number on the untangling. At each stage we fit the best straight-line
classifier — a "linear-separability probe" — and read its accuracy: how well a line splits that
representation. On the raw input it manages only about **0.88**; the crescents really are tangled.
The **first** hidden layer already lifts that to about **0.97**, and the second stays about there. On
this problem almost all the untangling happens in a *single* remap — the extra layer mostly
consolidates it. (That fits the single wide layer above, which reached the same ~0.96.) The
*mechanism* — remap until a line works — is real; here a single layer is enough.

One honest caveat: this probe is measured on the *training* points, so it is a *descriptive* read on
the representation, not a generalization claim. The honest accuracy story — with a held-out test set
— is what we turn to next.

## A fair question: is it the depth, or only more units?

Before we celebrate depth, look again at the printout above: a single **wide** layer (`2 → 8 → 2`)
also untangles the moons, reaching about **0.96** on its own. For a problem this small, depth is not
*required* — extra width in one layer does the job too.

That is the honest tension. Stacking layers untangles the moons — but so can one wide layer, and we
saw the deep net's *first* layer already do most of the work. So when we move to a real dataset we
must compare carefully: is depth buying us something that width alone would not?

## The honest test: depth vs width on real data

Let's put depth to a fair test on **MNIST** — 28×28 grayscale images of handwritten digits,
flattened to 784 pixels, ten classes. We use a 10,000-image training subset and a 5,000-image test
set, small enough to train by hand in numpy.

To separate "more layers" from "more units", we hold the **unit budget fixed** and compare three
networks:

- **50 units**, one hidden layer — a small baseline;
- **448 units**, one hidden layer — *wide*;
- **256 + 128 + 64 = 448 units**, three hidden layers — *deep*, the **same budget** as the wide one.

If depth itself helps, the deep network should beat the wide one: they have the same number of units.

In [ ]:
# One-time fetch (logged, never silenced), then a small subset we can train by hand.
print("Fetching MNIST (one-time download, then cached locally)...")
mnist = fetch_openml("mnist_784", version=1, as_frame=False)
X_all = mnist.data.astype(np.float64)
y_all = mnist.target.astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(
    X_all, y_all, train_size=10_000, test_size=5_000, stratify=y_all, random_state=SEED
)
X_tr, X_te = X_tr / 255.0, X_te / 255.0  # pixel intensities to [0, 1]
print(f"train {X_tr.shape}   test {X_te.shape}")

architectures = {
    "50 (shallow)": [784, 50, 10],
    "448 (wide)": [784, 448, 10],
    "256-128-64 (deep)": [784, 256, 128, 64, 10],
}
mnist_acc = {}
for name, sizes in architectures.items():
    net, _ = train(sizes, X_tr, y_tr, hidden="relu", lr=0.2, epochs=60, batch_size=128)
    mnist_acc[name] = (forward(net, X_te, "relu")[-1].argmax(1) == y_te).mean()
    print(f"  {name:18}: test accuracy {mnist_acc[name]:.4f}")

In [ ]:
names = list(mnist_acc.keys())
values = [mnist_acc[n] for n in names]
bar_colors = [colors.COLORS["muted"], colors.CLASS_CYCLE[1], colors.CLASS_CYCLE[3]]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(names, values, color=bar_colors, edgecolor=colors.COLORS["text"], linewidth=0.6)
for bar, value in zip(bars, values, strict=True):
    ax.text(
        bar.get_x() + bar.get_width() / 2, value, f"{value:.4f}",
        ha="center", va="bottom", color=colors.COLORS["text"],
    )
ax.set_ylim(0.90, 0.97)  # zoomed in: the differences are small (see "Read the figure")
ax.set_ylabel("test accuracy")
ax.set_title("Depth vs accuracy on MNIST\n(the last two share a 448-unit budget)")
plt.show()

**Read the figure** — note the zoomed-in y-axis; the differences are small. Going from 50 to 448
units lifts test accuracy by about **+1 point** (≈ 0.943 → 0.954): more capacity helps. But
arranging those same 448 units into a **deep** stack instead of a wide layer adds only about **+0.2
points** (≈ 0.954 → 0.956) — a near-wash. On these flat pixels the **capacity** (the number of
units) did the real work; **depth by itself barely moved the needle**. (scikit-learn's own
`MLPClassifier` tells the same story — roughly 0.94 → 0.95 across these shapes.)

## Why didn't depth win big here?

Depth untangled the moons dramatically — so why is it nearly a wash on MNIST? Because a dense network
throws away the one thing that makes images special: their **structure**. Flattened to 784 numbers,
pixel 1 and pixel 784 are treated as unrelated coordinates; the network has no idea they sit next to
each other on a grid. There is no spatial hierarchy for depth to climb.

The **vivid** version of "depth is a hierarchy" — early layers finding edges, later layers assembling
them into parts, then whole objects — is real, but it needs an architecture that **respects the
grid**: a **convolutional network (CNN)**. That is the horizon we point to in notebook 10. On flat,
tabular-style pixels, a deep dense network is honest, modest, and a little redundant with a wide one.

## A crack that opens as we go deeper

Two or three layers trained without trouble here. But the bet of deep learning is *many* layers —
ten, fifty, a hundred — and there the naive approach we have been using quietly breaks: stack enough
layers and the network stops learning, its loss curve flat from the very first epoch.

Why? The backward pass multiplies one factor per layer. Across many layers those factors compound,
and the gradient signal either **collapses toward zero** or **blows up**. That is the **vanishing and
exploding gradient** problem — the pivot of this chapter. Notebook 3 builds it by hand and measures
it; notebook 4 gives the first fix.

## What you built

- You **generalized** the notebook-1 network to any number of layers — the same forward pass,
  backward pass, and training loop, now looped over a list of layers.
- You **saw** depth at work: an all-width-2 network remapped two tangled moons into a linearly
  separable representation — most of it in the very first layer — and you measured that with a probe.
- You **measured** depth's honest gain on MNIST — and found that at a fixed unit budget, width did
  most of the lifting while depth-by-itself was nearly a wash.
- You learned **why**: flat pixels carry no spatial hierarchy for depth to exploit (that is a CNN's
  job), and you met the crack — vanishing/exploding gradients — that opens as networks get truly deep.

The reference network from notebook 1 is now a *deep* network. Next we find out why training a deep
one is hard.

## Your turn

1. **(warm-up)** Add a third hidden layer to the moons untangler (`2 → 2 → 2 → 2 → 2`) and re-train.
   Does it still untangle the moons? Does it train as easily as the two-layer version?
2. **(core)** Change the MNIST architectures — try `(128, 64, 32)` against a wide `(224,)` at the
   same unit budget. Does depth ever beat width by a clear margin? Re-read the accuracies.
3. **(reach)** Run the linear-separability probe layer by layer on the **deep MNIST** network (feed
   each hidden layer's activations to the probe). Does separability climb with depth there too, even
   though the final accuracy barely changes?

## Where this goes next

You have seen what depth promises — composing representations — and measured that the promise is
modest until the architecture matches the data. Notebook 3 confronts the obstacle head-on: as
networks get deep, the gradients that train them vanish or explode. Understanding that one mechanism
is the key that unlocks everything after it — initialization, normalization, and the modern recipe.

## References

- Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning*, chapter 6 (deep feedforward
  networks). MIT Press. https://www.deeplearningbook.org
- Olah, C. (2014). *Neural Networks, Manifolds, and Topology.*
  https://colah.github.io/posts/2014-03-NN-Manifolds-Topology/ — the intuition that a network
  untangles data by remapping it.
- Cybenko, G. (1989). Approximation by superpositions of a sigmoidal function. *Mathematics of
  Control, Signals and Systems* 2:303–314. https://doi.org/10.1007/BF02551274 — universal
  approximation (existence; recap from chapter 11).
- LeCun, Y., Bottou, L., Bengio, Y., & Haffner, P. (1998). Gradient-based learning applied to
  document recognition. *Proceedings of the IEEE* 86:2278–2324. https://doi.org/10.1109/5.726791 —
  convolutional networks (the horizon for the vivid feature hierarchy).

*Previous:* **12.1 — a neural network from scratch in numpy.**  *Next:* **12.3 — vanishing &
exploding gradients.**